[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/estimation/kalman/kalman.ipynb)

# Tracking a Pose with a Kalman Filter

A vehicle drives a meandering path in the plane. Its own odometry, the commands it gave, drifts further from the truth with every step; a few times along the way it gets a noisy reading of its whole pose, position and heading. A Kalman filter blends the two: it carries its best guess of the pose forward with the commands, keeps track of how uncertain that guess has become, and at each reading moves toward it by as much as the two uncertainties warrant.

The pose is a motor, and an error in it is a small twist on its right. The uncertainty is a covariance of those twists, typed as a map from a line, a linear readout of a twist through their pairing, to the twist correlated with it, `Twist <- Line`. Being a map, it moves through each step of the drive the way any map does, its readout pulled through the step and its twist pushed back, `step << sigma(step >> Line)`. The gain at a reading is a ratio of two such maps, and the position ellipse is the covariance read through the map from twists to the motion of the vehicle's centre.

In the matrix notation of the extended Kalman filter the covariance reads as a three-by-three matrix $P$, propagated as $F P F^{\mathsf T} + Q$ with the adjoint representation $F$ of each step and updated with the gain $K = P (P + R)^{-1}$.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
from collections.abc import Iterator

import numpy as np

from numga import Extensor
from numga import NumpyContext
from numga.algebras import PGA2D as ga
from examples.animation import show_animation
from examples.estimation.kalman import render
from examples.estimation.kalman.core import sample

np.set_printoptions(precision=4, suppress=True)

mv = NumpyContext(ga).multivector
Point = ga.gatype.antivector()
Motor = ga.gatype.rotor()
# A pose perturbation, the twist that carries the estimate to the truth.
Twist = ga.gatype.bivector()
Line = ga.gatype.antibivector()
# A linear readout of a twist is a line; the covariance takes it to the correlated twist.
Covariance = ga.gatype((Twist, Line))         # Twist <- Line

## 1. Predicting and correcting

The filter has two steps. A prediction carries the estimate through one commanded step and moves the covariance with it, adding the motion noise. A correction at a reading takes the innovation, the twist from the estimate to the reading, the log of their relative motor; the gain takes it to the twist to apply, and the covariance shrinks by what the reading told. Both noises are covariances, built from the coordinate twists of translation and of rotation about the body's origin.

In [ ]:
def covariance(translation_std: float, rotation_std: float) -> Covariance:
    """Body-frame noise: isotropic translation, and rotation about the body origin."""
    translation = mv.yw * (mv.yw & Line) + mv.wx * (mv.wx & Line)   # [] Twist <- Line
    rotation = mv.xy * (mv.xy & Line)                               # [] Twist <- Line
    return translation * translation_std**2 + rotation * rotation_std**2


def predict(estimate: Motor, sigma: Covariance, step: Motor, motion_noise: Covariance) -> tuple[Motor, Covariance]:
    """Carry the estimate and its covariance through one body-frame step, adding the motion noise.

    Pose errors are right perturbations, `estimate * (delta * 0.5).exp()`, with sigma their
    covariance on twists. A perturbation on the right of the estimate is carried through the step by
    step << delta: the adjoint of the step is its sandwich, step << Twist.
    """
    # The covariance takes a readout of the perturbation to the twist correlated with it,
    # so it moves like any map: pull the readout through the step, push the twist back.
    return estimate * step, (step << sigma(step >> Line)) + motion_noise


def update(estimate: Motor, sigma: Covariance, measured: Motor, measurement_noise: Covariance) -> tuple[Motor, Covariance]:
    """Correct the estimate and its covariance by a noisy pose measurement, its error a right perturbation
    with the measurement noise as covariance."""
    # The innovation is the log of the relative motor, the gain is a ratio of covariance maps, and the
    # correction is exponentiated.
    innovation = (estimate.inverse() * measured).log() * 2
    gain = sigma((sigma + measurement_noise).inverse())
    return estimate * (gain(innovation) * 0.5).exp(), sigma - gain(sigma)

## 2. The drive

The vehicle drives at unit speed along its own x while its turn rate wanders as a sine. The true motion adds noise to every commanded step, dead reckoning follows the commands alone, and the filter predicts through every step and corrects after each group of them, at a reading.

In [ ]:
# The time step, the readings and the commanded steps between them, and the seed of the noise.
dt, n_readings, n_steps_per_reading, seed = 0.1, 12, 25, 2
# The commanded turn rate's amplitude and angular frequency.
turn_amplitude, turn_frequency = 0.9, 0.2
# Standard deviations of translation and rotation, of the motion per square root of time and of a reading.
motion_std = np.array([0.05, 0.12])
measurement_std = np.array([0.3, 0.1])

In [ ]:
def drive(initial: Motor, increments: Twist, motion_noise: Covariance, measurement_noise: Covariance,
          rng: np.random.Generator) -> Iterator[tuple[Motor, Motor, Motor, Covariance, Motor]]:
    """The drive, one state per commanded step and one more after each reading: the true pose, moved by each
    increment and a draw of the motion noise; the pose dead-reckoned from the commands alone; the filter's
    estimate and covariance, predicted through each step and corrected at each reading; and the latest
    reading, a noisy measurement of the true pose."""
    truth = dead = estimate = reading = initial
    sigma = covariance(0.0, 0.0)
    for segment in increments:
        for increment in segment:
            truth = truth * ((increment + sample(motion_noise, rng.normal(size=3))) * 0.5).exp()
            dead = dead * (increment * 0.5).exp()
            estimate, sigma = predict(estimate, sigma, (increment * 0.5).exp(), motion_noise)
            yield truth, dead, estimate, sigma, reading
        reading = truth * (sample(measurement_noise, rng.normal(size=3)) * 0.5).exp()
        estimate, sigma = update(estimate, sigma, reading, measurement_noise)
        yield truth, dead, estimate, sigma, reading


times = np.arange(n_readings * n_steps_per_reading) * dt
turn = turn_amplitude * np.sin(turn_frequency * times)
increments = ((mv.xy * turn - mv.wx) * dt).reshape(n_readings, n_steps_per_reading)   # [readings, steps] Twist
motion_noise = covariance(*(motion_std * np.sqrt(dt)))                         # [] Twist <- Line
measurement_noise = covariance(*measurement_std)                               # [] Twist <- Line
states = drive(mv.rotor(), increments, motion_noise, measurement_noise, np.random.default_rng(seed))
truth, dead, estimates, covariances, readings = (Extensor.stack(path) for path in zip(*states))   # [states] Motor, ..., Twist <- Line, Motor

In [ ]:
print("the motion noise per step, a map from lines to twists:\n", motion_noise.coefficient_table())
print("the covariance after the last reading:\n", covariances[-1].coefficient_table())

## 3. The position and its uncertainty

A twist moves the vehicle's centre by its commutator with it, a map from twists to point displacements; its adjugate takes a line to the readout of the twist that reads that displacement, so the covariance seen through it is a form on lines, the position covariance, whose principal lines are the axes of the position ellipse. The animation follows the drive: the true path, black, dead reckoning, dashed red, and the filter, blue, each with its vehicle, the readings so far, green crosses, and the filter's 2σ position ellipse, which grows between readings and shrinks at each.

In [ ]:
def position_ellipse(estimate: Motor, sigma: Covariance, origin: Point):
    """The estimated position, with the principal variances and axes of its uncertainty.

    A readout of position along a line l, `l & shift(delta)`, is a readout of the twist through the
    incidence pairing, so the covariance of those readouts is a form on lines. The readout takes
    only the lines that see a displacement, those through the origin, which the line metric
    measures fully, so against it the principal readouts are the ellipse axes.
    """
    here = estimate >> origin                                  # [readings] Point
    shift = Twist.commutator(here)(estimate >> Twist)          # [readings] Point <- Twist
    readout = shift.adjugate()                                 # [readings] Line <- Line
    position = readout & sigma(readout)                        # [readings] Scalar <- (Line, Line)
    variances, axes = position.eigh()                          # [readings, modes] Scalar, [readings, modes] Line
    return here, variances, axes


def errors(truth: Motor, dead: Motor, estimates: Motor):
    """The position errors of dead reckoning and of the filter: the distance between two points of unit
    weight is the norm of the line joining them."""
    origin = mv.w.dual()
    true_here = truth >> origin
    return (true_here & (dead >> origin)).norm(), (true_here & (estimates >> origin)).norm()


here, variances, axes = position_ellipse(estimates, covariances, mv.w.dual())  # [states] Point, [states, modes] Scalar, Line
dead_error, filtered_error = errors(truth, dead, estimates)                    # [states] Scalar
show_animation(render.animate_tracking(truth, dead, estimates, here, variances, axes, readings), "kalman", 40)

In [ ]:
print("mean position error, dead reckoning:", dead_error.mean(axis=0).to_array())
print("mean position error, filtered:      ", filtered_error.mean(axis=0).to_array())

In [ ]:
# checks
# Sparse noisy readings keep the filtered path far closer to the truth than the dead-reckoned one, and the
# covariance stays its own adjugate through the filter.
assert filtered_error.mean(axis=0).to_array() < 0.25 * dead_error.mean(axis=0).to_array()
np.testing.assert_allclose(covariances.adjugate().kernel, covariances.kernel, atol=1e-12)